# CTB ProSiT reproduction

This notebook loads the saved CTB Petri net and ProSiT parameter bundles and reproduces the numerical results reported in the thesis.

The confidential terminal event log is not included. The saved workload-aware baseline and the two what-if models are simulated again from the PKL bundles. Historical hold-out, state-ablation, drift, structural-repair, bottleneck, and capacity results are recalculated from the included per-seed or derived evidence. They cannot be regenerated from the confidential raw events.

## 1. Install the frozen environment

Open this notebook from the reproduction directory with a Python 3.11 kernel and choose **Run All**.

In [ ]:
%pip install -r requirements.txt --disable-pip-version-check -q

## Saved model files

The package follows ProSiT's documented save/load approach:

- PNML stores the Petri-net control flow.
- JSON is the readable parameter export produced by `SimulatorParameters.to_json()`.
- PKL stores the exact calibrated Python object used for the thesis simulations.

The PKL is required for exact reproduction because ProSiT 1.0.3 does not restore the empirical sample arrays stored in the calibrated CTB rule leaves from JSON. The notebook demonstrates the JSON API and reports this difference. Only load the verified PKL files supplied in this folder.

## 2. Verify and load the saved models

In [ ]:
from pathlib import Path
import pickle

import pandas as pd
import pm4py
from prosit import SimulatorParameters, SimulatorEngine

import reviewer_runner as rr

integrity = rr.verify_package_files()
print(f"Verified frozen files: {len(integrity)}")
print(rr.package_versions().to_string(index=False))

net, initial_marking, final_marking = pm4py.read_pnml(
    "models/ctb_inductive_miner.pnml"
)
print(
    f"Petri net: {len(net.places)} places, "
    f"{len(net.transitions)} transitions, {len(net.arcs)} arcs"
)

with open("models/params_baseline_rmg_max_concurrency_3.pkl", "rb") as handle:
    baseline = pickle.load(handle)

engine = SimulatorEngine(baseline)
print(f"Loaded executable baseline: {type(engine).__name__}")

models = rr.load_models()
print("\nSaved model configurations")
print(rr.model_summary(models).to_string(index=False))

print("\nModel contract checks")
print(rr.assert_model_contracts(models).to_string(index=False))

json_report = rr.export_and_reload_official_json(
    baseline, Path("outputs/json_api_demo.json")
)
print("\nProSiT JSON export/import check")
print(pd.Series(json_report).to_string())

## 3. Reconstruct the remaining thesis results

These tables are recomputed from the included ten-seed validation outputs and non-confidential derived evidence.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import reviewer_runner as rr

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

historical_summary, historical_contrasts = rr.reconstruct_historical_ablation()
historical_summary.to_csv(output_dir / "historical_ablation_summary.csv", index=False)
historical_contrasts.to_csv(output_dir / "historical_ablation_contrasts.csv", index=False)

print("Historical three-state ablation")
headline = historical_summary[historical_summary["metric"].isin([
    "case_turnaround_emd_min",
    "case_turnaround_sim_mean",
    "case_turnaround_sim_p90",
    "yard_service_time_emd_frequency_weighted_min",
    "yard_activity_rate_l1_error",
    "gate_only_cases",
])]
print(headline.to_string(index=False))

print("\nPaired state contrasts")
print(historical_contrasts.to_string(index=False))

evidence = rr.load_claim_evidence()

temporal = evidence["temporal_transfer"]
print("\nTemporal transfer")
print(pd.Series({
    "train_mean_turnaround_min": temporal["turnaround"]["train_mean_min"],
    "test_mean_turnaround_min": temporal["turnaround"]["test_mean_min"],
    "mean_shift_min": temporal["turnaround"]["mean_shift"]["difference_test_minus_train"],
    "mean_shift_ci95_lo": temporal["turnaround"]["mean_shift"]["ci95_lo"],
    "mean_shift_ci95_hi": temporal["turnaround"]["mean_shift"]["ci95_hi"],
    "p90_shift_min": temporal["turnaround"]["p90_shift"]["difference_test_minus_train"],
    "yard_service_weighted_emd_min": temporal["yard_service_frequency_weighted_wasserstein_min"],
}).to_string())

repair = evidence["structural_repair"]
print("\nStructural repair")
repair_table = pd.DataFrame([
    {"model": "discovered", **repair["before"]["test"]},
    {"model": "gate_only_restricted", **repair["after"]["test"]},
])
print(repair_table[[
    "model", "fitness", "precision", "generalization", "simplicity"
]].to_string(index=False))

print("\nBottleneck ranking")
print(evidence["bottleneck_ranking"].head(10).to_string(index=False))

capacity = evidence["capacity_pressure"]
print("\nRMG capacity pressure")
print(pd.Series({
    "highest_pressure_block": capacity["highest_pressure_block"],
    "baseline_nominal_utilization": capacity["highest_baseline_nominal_utilization"],
    "demand_plus_20_nominal_utilization": capacity["highest_scenario_nominal_utilization"],
    "multiplier_to_mean_saturation": capacity["smallest_multiplier_to_mean_saturation"],
    "blocks_with_minutes_above_capacity": capacity["blocks_with_observed_minutes_above_capacity"],
}).to_string())

print("\nScenario state-ablation contrasts")
state_ablation = evidence["scenario_state_ablation"]
state_ablation = state_ablation[state_ablation["metric"].isin([
    "mean_turnaround_min",
    "mean_rmg_service_min",
    "mean_rmg_pre_service_min",
    "mean_rmg_receive_service_min",
    "mean_rmg_delivery_service_min",
])]
print(state_ablation.to_string(index=False))

## 4. Rerun the saved baseline and what-if models

The default run uses 10 matched seeds, 3 saved models, and 17,892 cases per model. It can take about 30 minutes. Changing `RUN_FULL_SCENARIOS` to `False` runs a short mechanics test only.

In [ ]:
import reviewer_runner as rr

RUN_FULL_SCENARIOS = True
mode = "full" if RUN_FULL_SCENARIOS else "smoke"

output_dir = rr.run_saved_models(mode=mode)
print(f"Fresh scenario outputs: {output_dir}")

if RUN_FULL_SCENARIOS:
    comparison = rr.compare_with_frozen_results(output_dir)
    comparison.to_csv(
        output_dir / "comparison_with_thesis_results.csv", index=False
    )
    print(comparison.to_string(index=False))
    assert comparison["values_match"].all()
    print("Full reproduction passed: all scenario tables match the thesis results.")
else:
    print("Smoke test passed. It verifies execution but not the thesis values.")

## Interpretation

The rerun establishes that the saved model and its two interventions are executable and reproducible. It does not establish physical causal effects at the terminal. The model does not contain explicit container locations, crane trajectories, physical transit, or queue states.